# Notebook 2: Creating Agents Properly

**What you'll learn:** Agent instructions design, output types, model settings, structured output, and real agentic patterns — WITHOUT handoffs or guardrails.

---

## The Agentic Workflow Mindset

Traditional AI:
```
User → Prompt → LLM → Response → Done
```

Agentic AI (2025-2026 paradigm):
```
User → Agent → [Think → Act → Observe → Think → Act → ...] → Result
```

The key difference: **the agent decides HOW to complete the task, not just WHAT to say.**

In [2]:
uv add openai-agents pydantic

Note: you may need to restart the kernel to use updated packages.


g:\Desktop\lab-agents\.venv\Scripts\python.exe: No module named uv


---

## Ollama Provider Setup

The `openai-agents` SDK does **not** recognise the `ollama/` string prefix natively.
Instead, we point it at Ollama's OpenAI-compatible local endpoint using
`AsyncOpenAI` + `OpenAIChatCompletionsModel`.

> **Pre-requisites — run once in a terminal:**
> ```bash
> ollama serve          # start the Ollama server
> ollama pull llama3.2  # download the model (first time only)
> ```

In [3]:
from openai import AsyncOpenAI
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

# Point the SDK at Ollama's local OpenAI-compatible endpoint
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",   # Ollama ignores the key; any non-empty string works
)

# Reusable model object — pass this as model= in every Agent()
OLLAMA_MODEL = OpenAIChatCompletionsModel(
    model="llama3.2:latest",
    openai_client=ollama_client,
)

print("Ollama provider ready  →  model: llama3.2:latest")

Ollama provider ready  →  model: llama3.2:latest


---

## Agent Anatomy — Every Parameter Explained

```python
Agent(
    name="...",              # Identity (used in tracing/debugging)
    instructions="...",      # System prompt — the agent's brain
    model=OLLAMA_MODEL,      # Ollama model object (llama3.2:latest)
    tools=[...],             # Functions the agent can call
    output_type=MyModel,     # Force structured JSON output (Pydantic)
    handoffs=[...],          # Other agents it can delegate to (Notebook 4+)
    guardrails=[...],        # Input/output validation (Notebook 5+)
    model_settings=...,      # Temperature, top_p, max_tokens, etc.
)
```

---

## Instructions Design — The Most Important Skill

The `instructions` field is your agent's **system prompt**. It defines:
- WHO the agent is (role/persona)
- WHAT it should do (task)
- HOW it should behave (constraints/rules)
- WHEN to use tools (decision criteria)

In [4]:
from agents import Agent, Runner

# BAD instructions — too vague
bad_agent = Agent(
    name="Bad Agent",
    instructions="You are helpful.",
    model=OLLAMA_MODEL
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """,
    model=OLLAMA_MODEL
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Score: 8/10

Issues:

1. No Error Handling: The function does not handle cases when x or y is not a number. Consider adding try-except block to handle such cases.

2. Type Hints: The function definition does not include type hints for the parameters and return type.

Suggestions:

1. Add Error Handling:
   ```python
def add(x, y):
    try:
        return x + y
    except TypeError:
        raise TypeError("x and y must be numbers")
```

2. Type Hints:
   ```python
def add(x: int | float, y: int | float) -> int | float:
    return x + y
```

This adds robustness to the function, while also improving clarity and maintainability through explicit type hints.


---

## Model Settings — Control LLM Behavior

In [5]:
from agents import Agent, Runner, ModelSettings

# Creative agent — high temperature
creative_agent = Agent(
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model=OLLAMA_MODEL,
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model=OLLAMA_MODEL,
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


 CREATIVE:
 Amidst the golden glow of the Pakistani sun, the Badshahi Mosque emerges as a magnificent testament to the ingenuity and grandeur of the Mughal Empire. Located in the heart of Lahore, this iconic mosque is a breathtaking blend of Islamic, Persian, and Indian architectural styles, set amidst the tran

 PRECISE:
 The Badshahi Mosque (Urdu: بادشاہی مسجد) is a Mughal-era mosque located in Lahore, Pakistan. It was built in 1673 by Mughal Emperor Aurangzeb during the reign of the Mughal Empire.

The mosque is considered one of the largest and most impressive in the world, with a capacity to accommodate up to 80,


---

## Structured Output — Force JSON Responses with Pydantic

This is a **game-changer** for production agents. Instead of parsing messy text, get clean typed data.

In [6]:
from pydantic import BaseModel, Field
from agents import Agent, Runner


# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    model=OLLAMA_MODEL,
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

OPENAI_API_KEY is not set, skipping trace export


Title:          Lex Fridman Podcast Review
Host:           Lex Fridman
Rating:         4.5/10
Genre:          Interviews
Tech Level:     Advanced
Summary:        In-depth conversations with AI and ML experts and thought leaders, exploring the intersection of technology and humanity
Recommended:    Yes


### Why Structured Output Matters for Agentic Workflows:

```
Without output_type:  "The movie is great, I'd give it 8/10..." → Parse with regex? 
With output_type:     MovieReview(rating=8.0, ...)              → Use directly! 
```

In multi-agent systems, structured output lets one agent's output become another agent's input **reliably**.

---

## Real-World Pattern: E-Commerce Product Classifier Agent

**Scenario:** You run an online store. Customers describe products they want. Your agent classifies and routes them.

In [7]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, ModelSettings


class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        description="Confidence score 0.0-1.0"
    )


classifier = Agent(
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    model=OLLAMA_MODEL,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1)
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")

OPENAI_API_KEY is not set, skipping trace export



Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: budget
   Search: 'affordable laptops for school' | Confidence: 80%


OPENAI_API_KEY is not set, skipping trace export



Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: low | Price: mid
   Search: 'birthday gift novel' | Confidence: 80%


OPENAI_API_KEY is not set, skipping trace export



Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: budget
   Search: 'phone charger' | Confidence: 90%


---

## Dynamic Instructions — Instructions as Functions

Instructions can be a **function** that returns a string, letting you inject runtime context:

In [8]:
from datetime import datetime
from agents import Agent, Runner


def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    name="Smart Scheduler",
    instructions=dynamic_instructions,  # <-- Pass function, not string!
    model=OLLAMA_MODEL
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Currently, it's 22:26, which is evening. Considering it's already quite late, for you, I would suggest winding down your day with a relaxing activity.

Why not take a few minutes to meditate, practice some gentle stretches, or read a book that you've been meaning to get to? You could also listen to some soothing music or nature sounds to calm your mind and prepare for a restful night's sleep.

How does that sound?


---

## Real-World Pattern: Customer Support Ticket Analyzer

**Industry Problem:** Thousands of support tickets daily. Agent analyzes sentiment, priority, and routes them.

In [9]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, ModelSettings


class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    model=OLLAMA_MODEL,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2)
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")

OPENAI_API_KEY is not set, skipping trace export



Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: angry
Priority:  P0-critical
Route to:  technical
Summary:   Our production is down due to API errors, and we need immediate assistance to resolve the issue.
Response:  I apologize for the inconvenience this has caused. I'm here to help you resolve the issue as quickly as possible. Our technical team is actively worki...


OPENAI_API_KEY is not set, skipping trace export



Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: neutral
Priority:  P3-low
Route to:  general
Summary:   You have been charged twice this month, and we're here to help.
Response:  Thank you for reaching out to us about this issue. I'm happy to help you resolve this. Can you please provide me with your order number or the specifi...


OPENAI_API_KEY is not set, skipping trace export



Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P3-low
Route to:  general
Summary:   Thank you for your feedback! We appreciate your enthusiasm for our product. We'll keep your suggestion in mind for future updates. However, our current development roadmap doesn't include a dark mode feature at this time. We'll be sure to consider it for a future release. If you have any other questions or need assistance with anything else, feel free to ask!
Response:  Thank you for your feedback! We appreciate your enthusiasm for our product. We'll keep your suggestion in mind for future updates. However, our curren...


---

## Agent Design Best Practices (2025-2026)

| Practice | Why |
|---|---|
| **Single Responsibility** | Each agent does ONE thing well |
| **Specific Instructions** | Vague = unpredictable, Specific = reliable |
| **Use output_type** | Structured data > parsing text |
| **Low temperature for tasks** | Deterministic results for business logic |
| **Dynamic instructions** | Inject runtime context (time, user data, state) |
| **Test with edge cases** | Empty input, adversarial input, long input |

---

## Summary

| Concept | Code |
|---|---|
| Model settings | `ModelSettings(temperature=0.1, max_tokens=500)` |
| Structured output | `Agent(output_type=MyPydanticModel)` |
| Dynamic instructions | `Agent(instructions=my_function)` |
| Production pattern | Classify → Route → Respond |

**Next:** Notebook 3 — Running all of this locally with Ollama (zero cost, full privacy).